# G6 — ETL KPM → tabela de features (economia de energia)

Grupo: G6 — Economia de energia (intenção simulada)
Pergunta: em que trechos do experimento a carga da célula está baixa o suficiente para, em tese,
justificar uma política de economia de energia — sem desligar nada de verdade?

Pipeline: `SQLite (bronze)` → tipagem/QC (`silver`) → coluna `baixa_carga` (regra de negócio G6) → CSV (`derived/`)

Adaptado de `code/notebooks/aula03_etl_kpm.ipynb` (material oficial da disciplina).


In [1]:
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)
import json
import sqlite3
from datetime import datetime, timezone
import pandas as pd

# Caminho relativo a analise-de-dados-aplicada-a-redes-de-telecomunicacoes/notebooks/
SAMPLE = Path("../../repo/data/code/datasets/kpm-ue-tp-sample")
DB = SAMPLE / "kpm.sqlite"
OUT = Path("../derived")
OUT.mkdir(parents=True, exist_ok=True)

FEATURE_KEYS = [
    "DRB.UEThpUl",
    "DRB.UEThpDl",
    "DRB.RlcSduDelayDl",
    "RRU.PrbTotUl",
]

# Limiar de "baixa carga" em RRU.PrbTotUl (%).
# Justificativa: baseline tem PRB constante (~2%, mediana=2.0, MAD=0 em model.json);
# stress tem PRB ~97-99%. Escolhemos um limiar conservador (10%) bem acima do
# platô de baseline e bem abaixo do platô de stress, para não classificar como
# "baixa carga" nenhum ponto de stress e para tolerar pequenas oscilações da
# fase recovery ao redor do baseline.
LIMIAR_PRB_BAIXA_CARGA = 10.0


## 1. Extract — SQLite (tabela `kpm_samples`)

In [2]:
con = sqlite3.connect(DB)
df = pd.read_sql(
    """
    SELECT run_id, phase, sample_index, ingested_at, source_path, payload_json
    FROM kpm_samples
    ORDER BY phase, sample_index
    """,
    con,
)
con.close()
print(df.shape)
display(df.head(3))


(100, 6)


,run_id,phase,sample_index,ingested_at,source_path,payload_json
0,ue-tp-20260804-174422,baseline,0,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,"{""DRB.RlcSduDelayDl"": 55.0, ""DRB.UEThpUl"": 4.4..."
1,ue-tp-20260804-174422,baseline,1,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,"{""DRB.RlcSduDelayDl"": 218.0, ""DRB.UEThpUl"": 3...."
2,ue-tp-20260804-174422,baseline,2,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,"{""DRB.RlcSduDelayDl"": 137.0, ""DRB.UEThpUl"": 3...."


## 2. Transform — tipagem, nulos, unidade lógica

In [3]:
records = []
for _, row in df.iterrows():
    try:
        payload = json.loads(row["payload_json"])
    except json.JSONDecodeError:
        payload = {}
    rec = {
        "run_id": row["run_id"],
        "phase": row["phase"],
        "sample_index": int(row["sample_index"]),
        "ingested_at": row["ingested_at"],
        "source_path": row["source_path"],
    }
    for key in FEATURE_KEYS:
        val = payload.get(key)
        rec[key] = float(val) if val is not None and val != "" else pd.NA
    records.append(rec)

features = pd.DataFrame.from_records(records)
# ordem didática de fases
phase_order = pd.CategoricalDtype(["baseline", "stress", "recovery"], ordered=True)
features["phase"] = features["phase"].astype(phase_order)
features = features.sort_values(["phase", "sample_index"]).reset_index(drop=True)

print("nulos por coluna:")
display(features.isna().sum())

print("duplicatas (run_id, phase, sample_index):", features.duplicated(subset=["run_id", "phase", "sample_index"]).sum())

display(features.head())


nulos por coluna:


run_id                 0
phase                  0
sample_index           0
ingested_at            0
source_path            0
DRB.UEThpUl            0
DRB.UEThpDl          100
DRB.RlcSduDelayDl      0
RRU.PrbTotUl           0
dtype: int64

duplicatas (run_id, phase, sample_index): 0


,run_id,phase,sample_index,ingested_at,source_path,DRB.UEThpUl,DRB.UEThpDl,DRB.RlcSduDelayDl,RRU.PrbTotUl
0,ue-tp-20260804-174422,baseline,0,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,4.46,<NA>,55.0,2.0
1,ue-tp-20260804-174422,baseline,1,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,218.0,2.0
2,ue-tp-20260804-174422,baseline,2,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,137.0,2.0
3,ue-tp-20260804-174422,baseline,3,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,171.0,2.0
4,ue-tp-20260804-174422,baseline,4,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,39.0,2.0


### 2.1 Regra de negócio do tema G6 — coluna `baixa_carga`

`baixa_carga = True` quando `RRU.PrbTotUl <= LIMIAR_PRB_BAIXA_CARGA`.

Esta é a base do KPI 1 (fração de tempo em baixa carga). O KQI 2 (vazão/delay médios
nessas janelas) é calculado a partir do mesmo flag no notebook de EDA.


In [4]:
features["baixa_carga"] = features["RRU.PrbTotUl"] <= LIMIAR_PRB_BAIXA_CARGA

display(features.groupby("phase", observed=True)["baixa_carga"].mean().round(3).rename("fracao_baixa_carga"))


phase
baseline    1.000
stress      0.017
recovery    0.950
Name: fracao_baixa_carga, dtype: float64

### 2.2 Registro de qualidade (QC)

In [5]:
from typing import Any

qc = {
    "rows": len(features),
    "phases": features["phase"].astype(str).value_counts().to_dict(),
    "null_fraction": features[FEATURE_KEYS].isna().mean().round(4).to_dict(),
    "duplicated_rows": int(features.duplicated(subset=["run_id", "phase", "sample_index"]).sum()),
    "timezone": "UTC (ingested_at em ISO 8601 com sufixo +00:00)",
    "limiar_prb_baixa_carga": LIMIAR_PRB_BAIXA_CARGA,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
qc["ok_has_baseline_stress"] = {"baseline", "stress"}.issubset(set[Any](features["phase"].astype(str)))
qc["ok_ue_thp_nonneg"] = bool((features["DRB.UEThpUl"].fillna(0) >= 0).all())
print(json.dumps(qc, indent=2))


{
  "rows": 100,
  "phases": {
    "stress": 60,
    "baseline": 20,
    "recovery": 20
  },
  "null_fraction": {
    "DRB.UEThpUl": 0.0,
    "DRB.UEThpDl": 1.0,
    "DRB.RlcSduDelayDl": 0.0,
    "RRU.PrbTotUl": 0.0
  },
  "duplicated_rows": 0,
  "timezone": "UTC (ingested_at em ISO 8601 com sufixo +00:00)",
  "limiar_prb_baixa_carga": 10.0,
  "generated_at": "2026-08-26T21:53:55.634231+00:00",
  "ok_has_baseline_stress": true,
  "ok_ue_thp_nonneg": true
}


## 3. Load — CSV (silver local)

In [6]:
csv_path = OUT / "kpm_features.csv"
qc_path = OUT / "etl_qc.json"

features.to_csv(csv_path, index=False)
qc_path.write_text(json.dumps(qc, indent=2), encoding="utf-8")

print("CSV:", csv_path.resolve())
print("QC:", qc_path.resolve())
display(features.groupby("phase", observed=True)[FEATURE_KEYS].median(numeric_only=True).round(2))


CSV: /Users/ar/Projects/pos/Modulo9/analise-de-dados-aplicada-a-redes-de-telecomunicacoes/derived/kpm_features.csv
QC: /Users/ar/Projects/pos/Modulo9/analise-de-dados-aplicada-a-redes-de-telecomunicacoes/derived/etl_qc.json


,DRB.UEThpUl,DRB.RlcSduDelayDl,RRU.PrbTotUl
phase,,,
baseline,3.72,0.0,2.0
stress,80023.68,158.9,99.0
recovery,3.72,0.0,2.0


## 4. Ligação com o lab (opcional)

```bash
# Regenerar bronze a partir do lab (se disponível)
cd ../../repo/data/code/oai-cn-gnb-nonrt-nearrt
./scripts/run_ue_tp_experiment.sh
```

Próximo notebook: `02_eda_kpm.ipynb` (consultas, indicadores preliminares e plots do CP1).
